# Overview

This notebook id dedicated for evaluating NER model on a benchmark dataset.

# Step 0 - Setup
Run the code below to mount your Google Drive and most of the necessary packages to carry out the evaluation

**Action:**
No code changes required. When prompted, connect your Google account

In [ ]:
%%capture
!pip install transformers
!pip install sentencepiece
!pip install seqeval
!pip install datasets

In [ ]:
%%capture
!cd /content/
!rm -rf ./CASM_utils/
!pip install git+https://github.com/ay94/multilingual-ner.git!pip install -e CASM_utils/
!cd /content/CASM_utils


import CASM_utils
import importlib
from CASM_utils import utils, ner
importlib.reload(ner)

In [ ]:
## Mount GDrive
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

import os
import pandas as pd

Mounted at /content/drive/


In [ ]:
# direct the file handler to the data folder
FOLDER = '/content/drive/MyDrive/CASM/FAST/German/NER/Benchmark'
fh = utils.FileHandler(FOLDER)

 # Read NER Dataset --> the output should be list of words corresponding to list of labels
 Read NER data class, it contains three different functionalities to read NER data.
  The NER data in the literature normally have consistent internal structure and flexible external structure.
  The internal structure is that it comes in word-label pair, this is consistent across all datasets.
  The external structure normally differ from dataset to another, which is divided to three main categories:
  - Data that comes in one text file, the read_ner_file function can be used in this case.
  - Data that comes in text files split into, train, val and test, this type you can either read individual file separately or put them all in one folder and read_ner_directory function.
  - Data that comes in directory where the directory contians various text files divide by topic (e.g, AQMAR), this type of data normally wikipedia articles that has been scraped and preprocessed into named entities structure.
  - Finally data available on huggingface and this can be loaded using load_dataset function and pass it to the read_dataset class method.
  
  Most of the datasets fall under one of these types if your data is different you can add function to this class dedicated to your data.

xtreme

In [ ]:
xtreme_label_map = {
    'O': 0, 'B-PER': 1, 'I-PER': 2,
    'B-ORG': 3, 'I-ORG': 4, 'B-LOC': 5, 'I-LOC': 6
}
xtreme = ner.ReadNERData()
xtreme_words, xtreme_labels = xtreme.read_dataset('xtreme', xtreme_label_map, lang='PAN-X.de')

/usr/local/lib/python3.10/dist-packages/datasets/load.py:1429: FutureWarning: The repository for xtreme contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/xtreme
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test Split


  0%|          | 0/10000 [00:00<?, ?it/s]

germeval_14

In [ ]:
germeval_14_label_map = {
    'O': 0,
    'B-LOC': 1, 'I-LOC': 2, 'B-LOCderiv': 3, 'I-LOCderiv': 4, 'B-LOCpart': 5, 'I-LOCpart': 6,
    'B-ORG': 7, 'I-ORG': 8, 'B-ORGderiv': 9, 'I-ORGderiv': 10, 'B-ORGpart': 11, 'I-ORGpart': 12,
    'B-OTH': 13, 'I-OTH': 14, 'B-OTHderiv': 15, 'I-OTHderiv': 16, 'B-OTHpart': 17, 'I-OTHpart': 18,
    'B-PER': 19, 'I-PER': 20, 'B-PERderiv': 21, 'I-PERderiv': 22, 'B-PERpart': 23, 'I-PERpart': 24
}



germeval_14 = ner.ReadNERData()
germeval_14_words, germeval_14_labels = germeval_14.read_dataset('germeval_14', germeval_14_label_map)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating test Split


  0%|          | 0/5100 [00:00<?, ?it/s]

### Check for dataset alignment
The first thing to do after loading the data is to check that it is aligned with the standard annotation scheme using check_labels function. NER datasets have various annotation schemes, the standard one we are interested in is the conll annotation scheme where the data should be divided into, *LOC*, *PERS*, *ORG*, *MISC* entities and each entity have BI boudary (e.g, B-LOC, I-LOC) and outside named entity O. This is the standard annotation scheme we are aiming for and some dataset comes with fine grained annotations or even different labels. This requires realigning the dataset labels to the standard scheme by defining a dataset label alignment dictionary and use the align_dataset function.

xtreme

In [ ]:
ner.check_labels(xtreme_labels)

{'B-LOC', 'B-ORG', 'B-PER', 'I-LOC', 'I-ORG', 'I-PER', 'O'}

collection3

In [ ]:
print(ner.check_labels(germeval_14_labels))
# Dataset Label Map Alignment to LOC, ORG, PERS, MISC
germeval_14_label_alignment = {
            "B-LOCderiv": "B-LOC", "I-LOCderiv": "I-LOC",
            "B-LOCpart": "B-LOC", "I-LOCpart": "I-LOC",
            "B-ORGderiv": "B-ORG", "I-ORGderiv": "I-ORG",
            "B-ORGpart": "B-ORG", "I-ORGpart": "I-ORG",
            "B-PERderiv": "B-ORG", "I-PERderiv": "I-ORG",
            "B-PERpart": "B-ORG", "I-PERpart": "I-ORG",
            "B-OTHderiv": "O", "I-OTHderiv": "O",
            "B-OTHpart": "O", "I-OTHpart": "O",
            "B-OTH": "O", "I-OTH": "O",

}

# Align the dataset labels to the standard labels
germeval_14_labels = ner.align_dataset(germeval_14_labels, germeval_14_label_alignment)
print(ner.check_labels(germeval_14_labels))


{'B-ORGderiv', 'I-LOCderiv', 'B-PERpart', 'I-LOC', 'B-OTHpart', 'B-PERderiv', 'B-LOC', 'I-OTH', 'B-PER', 'B-OTH', 'B-LOCderiv', 'B-ORGpart', 'I-ORG', 'B-OTHderiv', 'I-ORGpart', 'O', 'I-PER', 'B-LOCpart', 'I-PERpart', 'B-ORG'}
{'B-PER', 'I-ORG', 'O', 'B-ORG', 'I-PER', 'I-LOC', 'B-LOC'}


# Model Evaluation
Model evaluation is dvided into three steps:
- Loading the model using get_model funtion
- Generating the evaluation benchmark using generate_evaluation_data function
- Apply the model to the benchmakr and compute the performance using eval_fn

All of these steps can be achieved by calling evaluate_model function. It is worth noting that all models comes with their own labeling scheme and some models have different annotation scheme from the standard one we are using, this requires using label alignment dictionary to align the model's output.

In [ ]:
model_name = "julian-schelb/roberta-ner-multilingual"
model_name_output = 'julian-roberta-Wikiann'
model_evaluation = ner.ModelEvaluation(model_name, germeval_14_label_alignment)

In [ ]:
model_evaluation.tokenizer._tokenizer.pre_tokenizer

In [ ]:
fh.create_folder(f'outputs/{model_name_output}')

Folder 'outputs/julian-roberta-Wikiann' already exists.


In [ ]:
model_evaluation.model.config.id2label

{0: 'O',
 1: 'B-PER',
 2: 'I-PER',
 3: 'B-ORG',
 4: 'I-ORG',
 5: 'B-LOC',
 6: 'I-LOC'}

#### xtreme

In [ ]:
data_name = "xtreme"
xtreme_evaluation_output = model_evaluation.evaluate_model(xtreme_words, xtreme_labels)

  0%|          | 0/625 [00:00<?, ?it/s]

In [ ]:
xtreme_seqeval = xtreme_evaluation_output.get_classification('Seqeval')
xtreme_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.8723,0.8869,0.8796,4961
1,ORG,0.8085,0.8268,0.8176,4157
2,PER,0.9322,0.9206,0.9264,4750
3,micro,0.8730,0.8804,0.8767,13868
4,macro,0.8710,0.8781,0.8745,13868
5,weighted,0.8737,0.8804,0.8770,13868


In [ ]:
xtreme_sklearn = xtreme_evaluation_output.get_classification('Sklearn')
xtreme_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.8937,0.8811,0.8873,4961
1,B-ORG,0.8435,0.8196,0.8314,4157
2,B-PER,0.9542,0.9160,0.9347,4750
3,I-LOC,0.7954,0.8541,0.8237,2289
4,I-ORG,0.8875,0.8943,0.8909,6043
5,I-PER,0.9563,0.9494,0.9528,6792
6,O,0.9892,0.9923,0.9908,68654
7,accuracy,0.9633,97646,None,None
8,macro,0.9028,0.9009,0.9016,97646
9,weighted,0.9633,0.9633,0.9633,97646


In [ ]:
xtreme_seqeval.to_csv(
    fh.cr_fn(f'outputs/{model_name_output}/{data_name}-seqeval.csv'),
    index=False
)
xtreme_sklearn.to_csv(
    fh.cr_fn(f'outputs/{model_name_output}/{data_name}-sklearn.csv'),
    index=False
)

#### germeval_14

In [ ]:
data_name = "germeval_14"
germeval_14_evaluation_output = model_evaluation.evaluate_model(germeval_14_words, germeval_14_labels)

  0%|          | 0/319 [00:00<?, ?it/s]

In [ ]:
germeval_14_seqeval = germeval_14_evaluation_output.get_classification('Seqeval')
germeval_14_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.7524,0.6355,0.6890,2376
1,ORG,0.2096,0.6426,0.3161,1385
2,PER,0.8478,0.7102,0.7729,1639
3,micro,0.4673,0.6600,0.5472,5400
4,macro,0.6033,0.6628,0.5927,5400
5,weighted,0.6421,0.6600,0.6188,5400


In [ ]:
germeval_14_sklearn = germeval_14_evaluation_output.get_classification('Sklearn')
germeval_14_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.7839,0.6566,0.7146,2376
1,B-ORG,0.2310,0.6895,0.3460,1385
2,B-PER,0.8665,0.7010,0.7750,1639
3,I-LOC,0.4677,0.6124,0.5303,307
4,I-ORG,0.2490,0.8300,0.3831,706
5,I-PER,0.8256,0.9550,0.8856,912
6,O,0.9868,0.9432,0.9645,89174
7,accuracy,0.9267,96499,None,None
8,macro,0.6301,0.7697,0.6570,96499
9,weighted,0.9603,0.9267,0.9399,96499


In [ ]:
germeval_14_seqeval.to_csv(
    fh.cr_fn(f'outputs/{model_name_output}/{data_name}-seqeval.csv'),
    index=False
)
germeval_14_sklearn.to_csv(
    fh.cr_fn(f'outputs/{model_name_output}/{data_name}-sklearn.csv'),
    index=False
)
